# Kredi Kartı Dolandırıcılığı Tespiti

Önceki egzersizler, bir sinir ağının tüm farklı bileşenlerine daha yakından bakmanızı sağladı:
* sıralı (sequential) yoğun (Dense) bir Sinir Ağı mimarisi,
* derleme (compilation) yöntemi,
* modelin eğitilmesi (fitting).

Şimdi **çok fazla veri içeren** gerçek hayat bir veri seti üzerinde çalışalım!

**Veri seti: `Kredi Kartı İşlemleri (Credit Card Transactions)`**

Bu açık uçlu challenge’da, **kredi kartı işlemlerinden elde edilmiş verilerle** çalışacaksınız.

Bu veriler **hassas** olduğu için, toplam 31 sütundan yalnızca 3’ü bilinmektedir; geri kalanlar verileri **anonimleştirmek** amacıyla dönüştürülmüştür (aslında bunlar, orijinal verilerin **PCA projeksiyonlarıdır**).

Bilinen 3 sütun şunlardır:

* `TIME`: İşlemin, veri setindeki ilk işleme göre geçen süresi  
* `AMOUNT`: İşlem tutarı  
* `CLASS` (hedef değişkenimiz):
    * `0 : geçerli işlem`
    * `1 : sahte (fraud) işlem`

❓ **Soru** ❓ Veri setini indirerek başlayın:
* Kaggle üzerinden: [buradan](https://www.kaggle.com/mlg-ulb/creditcardfraud)
* veya bizim URL’mizden: [buradan](https://d32aokrjazspmn.cloudfront.net/materials/creditcard.csv)

Veriyi yükleyerek `X` ve `y` değişkenlerini oluşturun.

In [12]:
import pandas as pd

try:
    df = pd.read_csv('creditcard.csv')
    print("✅ Veri seti başarıyla yüklendi.")
    
    # X ve y'yi ayıralım
    X = df.drop('Class', axis=1)
    y = df['Class']
    
    print(f"Toplam Satır: {len(df)}")
except Exception as e:
    print(f"Hala bir sorun var: {e}")

✅ Veri seti başarıyla yüklendi.
Toplam Satır: 284807


In [13]:
# Özellikleri (X) ve Hedef Değişkeni (y) ayıralım
X = df.drop(columns=['Class'])
y = df['Class']

print(f"X (Özellikler) boyutu: {X.shape}")
print(f"y (Hedef) boyutu: {y.shape}")

X (Özellikler) boyutu: (284807, 30)
y (Hedef) boyutu: (284807,)


In [14]:
# Sınıf dağılımını rakamsal olarak gör
print(y.value_counts())

# Yüzdesel olarak gör
print("\nDolandırıcılık Oranı: %", (y.value_counts()[1] / len(y) * 100).round(4))

Class
0    284315
1       492
Name: count, dtype: int64

Dolandırıcılık Oranı: % 0.1727


In [15]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Time ve Amount sütunlarını ölçeklendiriyoruz
scaler = StandardScaler()
X['Amount'] = scaler.fit_transform(X[['Amount']])
X['Time'] = scaler.fit_transform(X[['Time']])

# Veriyi Eğitim ve Test setine bölelim (%80 Eğitim, %20 Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Veri ölçeklendi ve bölündü!")
print(f"Eğitim seti boyutu: {X_train.shape}")

Veri ölçeklendi ve bölündü!
Eğitim seti boyutu: (227845, 30)


## 1. Sınıfların yeniden dengelenmesi

In [16]:
# Sınıf dengesini kontrol edelim
pd.Series(y).value_counts(normalize=True)

Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64

☝️ Bu `fraud detection` (sahtekârlık tespiti) challenge’ında **sınıflar aşırı derecede dengesizdir**:
* %99.8 normal işlemler
* %0.2 sahte (fraud) işlemler

**Ciddi yeniden dengeleme (rebalancing) stratejileri uygulamazsak sahtekârlık vakalarını tespit edemeyiz!**

❓ **Soru** ❓
1. **Önce**, veri setinizden üç ayrı bölme oluşturun: `Train / Val / Test`.  
   Doğrulama (validation) ve test setlerinin **dengesiz** kalması son derece önemlidir; böylece modeli değerlendirirken gerçek koşullar korunur ve veri sızıntısı (data leakage) oluşmaz. Test setinizi bu notebook’un **en son hücresine kadar saklayın**!

&nbsp;

2. **İkinci olarak**, yalnızca eğitim setinizi (training set) yeniden dengeleyin. Birçok seçeneğiniz var:

- Azınlık sınıfını rastgele oversample etmek (düz NumPy fonksiyonlarıyla).  
  *(En iyi seçenek değildir; çünkü satırları kopyalayarak veri sızıntısı yaratır.)*
- <a href="https://machinelearningmastery.com/smote-oversampling-for-imbalanced-classification/">**`Synthetic Minority Oversampling Technique - SMOTE`**</a> kullanarak, mevcut gözlemleri ağırlıklandırıp yeni veri noktaları üretmek
- Buna ek olarak, çoğunluk sınıfını bir miktar küçültmek için  
  <a href="https://machinelearningmastery.com/random-oversampling-and-undersampling-for-imbalanced-classification/">**`RandomUnderSampler`**</a> da deneyebilirsiniz

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# 1. ADIM: Veriyi bölme (Test setini 30.000 sınırının üzerine çıkarmak için %20 ayırıyoruz)
# Bu bölme ile X_test yaklaşık 56.962 satır olacak.
X_train_temp, X_test, y_train_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. ADIM: Kalan %80'i kendi içinde Train ve Validation olarak bölüyoruz
X_train, X_val, y_train, y_val = train_test_split(X_train_temp, y_train_temp, test_size=0.2, random_state=42, stratify=y_train_temp)

# 3. ADIM: Ölçeklendirme (Scaling)
# Sadece X_train ile fit ediyoruz, diğerlerini sadece transform ediyoruz (Data Leakage önlemi)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 4. ADIM: Sadece Eğitim Setini Dengeleme (SMOTE)
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(f"Test Seti Boyutu: {len(X_test)} (30.000'den büyük olmalı)")
print(f"Eğitim Seti Dağılımı: {y_train_res.value_counts()}")

Test Seti Boyutu: 56962 (30.000'den büyük olmalı)
Eğitim Seti Dağılımı: Class
0    181961
1    181961
Name: count, dtype: int64


In [28]:
from imblearn.over_sampling import SMOTE
from collections import Counter

# Dengeleme öncesi durumu görelim
print(f"Dengeleme öncesi eğitim seti dağılımı: {Counter(y_train)}")

# SMOTE uyguluyoruz (Sadece eğitim verisine!)
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Sonucu kontrol edelim
print(f"Dengeleme sonrası eğitim seti dağılımı: {Counter(y_train_res)}")

Dengeleme öncesi eğitim seti dağılımı: Counter({0: 181961, 1: 315})
Dengeleme sonrası eğitim seti dağılımı: Counter({0: 181961, 1: 181961})


## 2. Sinir Ağı yinelemeleri

Sınıflarınızı yeniden dengelediğinize göre, test puanınızı optimize etmek için bir sinir ağı uyarlayın. Aşağıdaki ipuçlarını kullanmaktan çekinmeyin:

- Girişlerinizi normalleştirin!
    - Modelinizdeki ön işlemeyi “boru hattı” haline getirmek için tercihen model içinde bir [`Normalization`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Normalization) katmanı kullanın. 
    - Veya modelinizin dışında sklearn'in [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) öğesini kullanın, `X_train`, `X_val` ve `X_test` öğelerini uygulayın.
- Modelinizi aşırı uyumlu hale getirin, ardından aşağıdakileri kullanarak düzenleyin:
- Erken Durdurma kriterleri
- [`Dropout`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout) katmanları
    - veya [`düzenleyiciler`](https://www.tensorflow.org/api_docs/python/tf/keras/regularizers) katmanları
- 🚨 İzlemek istediğiniz metrikleri ve kullanmak istediğiniz kayıp fonksiyonunu dikkatlice düşünün!

In [29]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.metrics import Precision, Recall, AUC

# 1. Veriyi Ölçeklendirme (Sinir ağları için şart!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 2. Model Mimarisini Oluşturma
model = models.Sequential([
    # Giriş Katmanı ve İlk Gizli Katman
    layers.Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dropout(0.2), # Overfitting'i önlemek için nöronların %20'sini rastgele kapatır
    
    # İkinci Gizli Katman
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.2),
    
    # Üçüncü Gizli Katman
    layers.Dense(8, activation='relu'),
    
    # Çıkış Katmanı (İkili sınıflandırma için sigmoid)
    layers.Dense(1, activation='sigmoid')
])

# 3. Modeli Derleme (Metrik seçimi kritik!)
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy', 
        Precision(name='precision'), 
        Recall(name='recall'), 
        AUC(name='auc')
    ]
)

# 4. Erken Durdurma (Early Stopping)
# Validation loss 5 epoch boyunca iyileşmezse durdur ve en iyi ağırlıklara dön.
early_stop = callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True
)

# 5. Modeli Eğitme
history = model.fit(
    X_train_scaled, y_train_res,
    validation_data=(X_val_scaled, y_val),
    epochs=50,
    batch_size=1024, # Veri seti çok büyük olduğu için büyük bir batch size seçtik
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50


/home/semih/.pyenv/versions/3.12.9/envs/workintech/lib/python3.12/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


356/356 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9087 - auc: 0.9686 - loss: 0.2258 - precision: 0.9234 - recall: 0.8914 - val_accuracy: 0.9826 - val_auc: 0.9528 - val_loss: 0.0655 - val_precision: 0.0782 - val_recall: 0.8354
Epoch 2/50
356/356 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9750 - auc: 0.9964 - loss: 0.0746 - precision: 0.9766 - recall: 0.9733 - val_accuracy: 0.9847 - val_auc: 0.9455 - val_loss: 0.0455 - val_precision: 0.0878 - val_recall: 0.8354
Epoch 3/50
356/356 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9851 - auc: 0.9986 - loss: 0.0456 - precision: 0.9822 - recall: 0.9880 - val_accuracy: 0.9905 - val_auc: 0.9353 - val_loss: 0.0303 - val_precision: 0.1379 - val_recall: 0.8481
Epoch 4/50
356/356 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9906 - auc: 0.9992 - loss: 0.0313 - precision: 0.9867 - recall: 0.9945 - val_accuracy: 0.9930 - val_auc: 0.9259 - val_loss: 0.0248 - val_precision: 0.1791 - val_recall: 0.8481
Epoch 5/50
356/356 ━━━━━━━━━━━━━━━━━━━━ 1s 

### 🧪 Kodunu Test Et

Orijinal dengesiz veri kümesinin (`X_test`, `y_test`) temsil edici bir örneğinde gerçek test performansınızın altında kalan sonuçları `precision` ve `recall` değişkenlerine kaydedin.

In [30]:
from sklearn.metrics import precision_score, recall_score, classification_report, confusion_matrix

# 1. Modelin tahminlerini al (Sigmoid çıktısı olduğu için 0.5 eşiğini kullanalım)
y_pred_probs = model.predict(X_test_scaled)
y_pred = (y_pred_probs > 0.5).astype(int)

# 2. Precision ve Recall değerlerini hesapla
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print(f"--- TEST SONUÇLARI ---")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print("-" * 25)

# 3. Detaylı rapor (F1-score ve destek değerlerini görmek için)
print("\nSınıflandırma Raporu:")
print(classification_report(y_test, y_pred))

# 4. Karmaşıklık Matrisi (Hangi işlemi ne olarak tahmin etti?)
print("\nKarmaşıklık Matrisi (Confusion Matrix):")
conf_matrix = confusion_matrix(y_test, y_pred)
print(conf_matrix)

1781/1781 ━━━━━━━━━━━━━━━━━━━━ 1s 551us/step
--- TEST SONUÇLARI ---
Precision: 0.4315
Recall:    0.8673
-------------------------

Sınıflandırma Raporu:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.43      0.87      0.58        98

    accuracy                           1.00     56962
   macro avg       0.72      0.93      0.79     56962
weighted avg       1.00      1.00      1.00     56962


Karmaşıklık Matrisi (Confusion Matrix):
[[56752   112]
 [   13    85]]


In [31]:
from nbresult import ChallengeResult

result = ChallengeResult('solution',
    precision=precision,
    recall=recall,
    fraud_number=len(y_test[y_test == 1]),
    non_fraud_number=len(y_test[y_test == 0]),
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/semih/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/semih/code/Semih0799/S18D3-S-Data-credit-card-challenge/tests
plugins: anyio-4.8.0, typeguard-4.4.2
collecting ... collected 2 items

test_solution.py::TestSolution::test_is_score_good_enough PASSED         [ 50%]
test_solution.py::TestSolution::test_is_test_set_representative PASSED   [100%]

============================== 2 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/solution.pickle

git commit -m 'Completed solution step'

git push origin master



## 🏁 İsteğe bağlı: Bu zorluk için Google'ın çözümünü okuyun
Bu oturumdaki tüm zorlukları tamamladığınız için tebrikler!

Son olarak, Google'ın kendi çözümünü doğrudan [Colab'da buradan](https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/tutorials/structured_data/imbalanced_data.ipynb) okuyun. 

İlginç teknikler ve en iyi uygulamaları keşfedeceksiniz.